# 04 · Belief Knowledge Graph (BKG) Encoding

Converts the network estimation outputs into a **Belief Knowledge Graph (BKG)**
encoded as CSV node/relationship tables and a Neo4j Cypher import script.

## Schema

```
Variable ←──INSTANCE_OF── NodeInstance ──────────────────┐
                               ↑                          │
                    HAS_NODE_INSTANCE                     │
                               │                          │
NetworkInstance ──HAS_CONTEXT──→ Context                  │
     │                                                    │
     │   (gamma, cor_method stored as properties)         │
     │                                                    │
     └──HAS_EDGE_INSTANCE──→ EdgeInstance ──FROM/TO──→ NodeInstance
```

## Scope

- Only the main specification `g05` (γ = 0.50, Pearson) is stored — **6 NetworkInstance nodes**
  (3 waves × 2 trust strata).
- `g075`, `g05_thr`, and `g05_spearman` are robustness diagnostics and are **not** encoded in the BKG.
- Estimation metadata (gamma, cor_method) is stored as properties on NetworkInstance.

## Outputs  (`../results/networks/kg_exports/`)

| Type | File |
|------|------|
| Node tables | `variables.csv`, `contexts.csv`, `network_instances.csv`, `node_instances.csv`, `edge_instances.csv` |
| Relationship tables | `rel_has_context.csv`, `rel_network_has_node_instance.csv`, `rel_node_instance_of_variable.csv`, `rel_network_has_edge_instance.csv`, `rel_edge_instance_from.csv`, `rel_edge_instance_to.csv` |
| Import script | `import_kg.cypher` |


In [31]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# ── Debug: where are we? ─────────────────────────────────────────────────────
print("Current working directory:", os.getcwd())
print()

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = Path("..")
NETWORK_DIR = BASE_DIR / "results" / "networks"
KG_DIR      = NETWORK_DIR / "kg_exports"

print("BASE_DIR resolves to:   ", BASE_DIR.resolve())
print("NETWORK_DIR resolves to:", NETWORK_DIR.resolve())
print("NETWORK_DIR exists?     ", NETWORK_DIR.resolve().exists())
print("KG_DIR resolves to:     ", KG_DIR.resolve())
print()

if not NETWORK_DIR.resolve().exists():
    raise FileNotFoundError(
        f"NETWORK_DIR not found: {NETWORK_DIR.resolve()}.\n"
        "Run 02_ggm_estimation and 03_results_extraction_R first.\n"
        f"Current working directory is: {os.getcwd()}"
    )

KG_DIR.mkdir(parents=True, exist_ok=True)
print("KG_DIR ready:", KG_DIR.resolve())

# ── KG scope constants ────────────────────────────────────────────────────────
KG_SPECS = ["g05"]
print("KG_SPECS:", KG_SPECS)

Current working directory: c:\Users\Ciinz\Desktop\ThesisFinal\Notebooks

BASE_DIR resolves to:    C:\Users\Ciinz\Desktop\ThesisFinal
NETWORK_DIR resolves to: C:\Users\Ciinz\Desktop\ThesisFinal\Results\networks
NETWORK_DIR exists?      True
KG_DIR resolves to:      C:\Users\Ciinz\Desktop\ThesisFinal\Results\networks\kg_exports

KG_DIR ready: C:\Users\Ciinz\Desktop\ThesisFinal\Results\networks\kg_exports
KG_SPECS: ['g05']


In [32]:
# ── Load upstream results tables ─────────────────────────────────────────────
REQUIRED_FILES = {
    "edges_all.csv"   : NETWORK_DIR / "edges_all.csv",
    "node_metrics.csv": NETWORK_DIR / "node_metrics.csv",
    "edge_metrics.csv": NETWORK_DIR / "edge_metrics.csv",
}

missing = [k for k, v in REQUIRED_FILES.items() if not v.exists()]
if missing:
    raise FileNotFoundError(
        "Required upstream files are missing: " + ", ".join(missing) +
        "\nRun 03_results_extraction_R first."
    )

edges_all    = pd.read_csv(REQUIRED_FILES["edges_all.csv"])
node_metrics = pd.read_csv(REQUIRED_FILES["node_metrics.csv"])
edge_metrics = pd.read_csv(REQUIRED_FILES["edge_metrics.csv"])

print("edges_all   :", edges_all.shape)
print("node_metrics:", node_metrics.shape)
print("edge_metrics:", edge_metrics.shape)

edges_all   : (442, 22)
node_metrics: (192, 17)
edge_metrics: (442, 29)


In [33]:
# ── Validate required columns in upstream tables ──────────────────────────────
REQUIRED_EDGE_COLS = {
    "wave","trust_stratum","n_cc","spec","spec_id","gamma_ebic",
    "context_id","network_id","edge_id","from","to","edge_a","edge_b",
    "weight","abs_weight","from_domain","to_domain","from_role","to_role",
}
REQUIRED_NODE_COLS = {
    "wave","trust_stratum","n_cc","spec","spec_id","gamma_ebic",
    "context_id","network_id","node","node_id","degree","strength","domain","role",
}

missing_edge = sorted(REQUIRED_EDGE_COLS - set(edge_metrics.columns))
missing_node = sorted(REQUIRED_NODE_COLS - set(node_metrics.columns))

if missing_edge:
    raise ValueError(f"edge_metrics is missing columns: {missing_edge}")
if missing_node:
    raise ValueError(f"node_metrics is missing columns: {missing_node}")

# Confirm KG specs are present
specs_found = set(edge_metrics["spec"].unique())
specs_missing = set(KG_SPECS) - specs_found
if specs_missing:
    raise ValueError(
        f"KG specs {specs_missing} not found in edge_metrics. "
        f"Available specs: {specs_found}"
    )

print("edge_metrics columns: OK")
print("node_metrics columns: OK")
print("KG specs present    :", sorted(specs_found & set(KG_SPECS)))

edge_metrics columns: OK
node_metrics columns: OK
KG specs present    : ['g05']


In [34]:
# ── Variable nodes ────────────────────────────────────────────────────────────
# Variables are stable across all network instances (wave/stratum/spec).
# We derive them from node_metrics and verify they are truly constant.

variables = (
    node_metrics[["node_id","node","domain","role"]]
    .drop_duplicates()
    .rename(columns={"node": "name"})
    .sort_values(["domain","name"])
    .reset_index(drop=True)
)

if not variables["node_id"].is_unique:
    raise ValueError(
        "Duplicate node_id values in variables table:\n"
        + str(variables[variables.duplicated("node_id", keep=False)])
    )

# Each node_id should map to exactly one (name, domain, role) triple
inconsistent = (
    node_metrics.groupby("node_id")[["node","domain","role"]]
    .nunique()
    .pipe(lambda d: d[d.max(axis=1) > 1])
)
if not inconsistent.empty:
    raise ValueError(
        "node_id values map to multiple (name, domain, role) combinations:\n"
        + str(inconsistent)
    )

print("Variable nodes:", len(variables))
display(variables)

Variable nodes: 8


,node_id,name,domain,role
0,AFF_FEAR,AFF_FEAR,affect,non_focal
1,AFF_WORRY,AFF_WORRY,affect,non_focal
2,WORRY_HEALTH_SYSTEM,WORRY_HEALTH_SYSTEM,affect,non_focal
3,USE2_AVOID,USE2_AVOID,behavior,non_focal
4,USE2_HANDWASH20,USE2_HANDWASH20,behavior,non_focal
5,USE2_MASK,USE2_MASK,behavior,non_focal
6,USE2_SPACE150,USE2_SPACE150,behavior,non_focal
7,SEVERITY,SEVERITY,belief,focal_belief


In [35]:
# ── Context and NetworkInstance tables ────────────────────────────────────────
# Restricted to KG_SPECS only (main specification: g05, Pearson, γ=0.5).
# Estimation metadata stored as properties on NetworkInstance (no separate GammaSpec node).
#
# global_strength (GS = sum of absolute edge weights per network) is computed
# from edges_all (upper-triangle, each edge appears once) and stored as a
# NetworkInstance property so it is available for Cypher queries without
# re-aggregation inside Neo4j.

edge_metrics_kg = edge_metrics[edge_metrics["spec"].isin(KG_SPECS)].copy()
edges_all_kg    = edges_all[edges_all["spec"].isin(KG_SPECS)].copy()

gs_per_network = (
    edges_all_kg
    .groupby("network_id", as_index=False)["abs_weight"]
    .sum()
    .rename(columns={"abs_weight": "global_strength"})
)

contexts = (
    edge_metrics_kg[["context_id","wave","trust_stratum","n_cc"]]
    .drop_duplicates()
    .assign(
        trust_variable          = "TRUST_RKI",
        trust_grouping_rule     = "within-wave terciles",
        middle_tercile_excluded = True,
    )
    .sort_values(["wave","trust_stratum"])
    .reset_index(drop=True)
)

network_instances = (
    edge_metrics_kg[["network_id","context_id","spec","gamma_ebic",
                      "wave","trust_stratum","n_cc"]]
    .drop_duplicates()
    .merge(gs_per_network, on="network_id", how="left")
    .assign(
        gamma      = lambda d: d["gamma_ebic"],
        cor_method = "pearson",
    )
    .sort_values(["wave","trust_stratum"])
    .reset_index(drop=True)
)

# ── Assertions ────────────────────────────────────────────────────────────────
if not contexts["context_id"].is_unique:
    raise ValueError("Duplicate context_id values.")
if not network_instances["network_id"].is_unique:
    raise ValueError("Duplicate network_id values.")

n_gs_na = network_instances["global_strength"].isna().sum()
if n_gs_na > 0:
    raise ValueError(
        f"{n_gs_na} network instance(s) have NA global_strength. "
        "Check that edges_all covers all KG network_ids."
    )

# One instance per context (6 total).
if len(network_instances) != len(contexts):
    raise ValueError(
        f"Expected {len(contexts)} network instances (one per context), "
        f"found {len(network_instances)}."
    )

print(f"Contexts          : {len(contexts)}")
print(f"Network instances : {len(network_instances)}  (one per context)")
print(f"global_strength   : [{network_instances['global_strength'].min():.4f}, "
      f"{network_instances['global_strength'].max():.4f}]")
display(network_instances)


Contexts          : 6
Network instances : 6  (one per context)
global_strength   : [2.3034, 3.2650]


,network_id,context_id,spec,gamma_ebic,wave,trust_stratum,n_cc,global_strength,gamma,cor_method
0,net_g05_w55_high,ctx_w55_high,g05,0.5,55,high,309,2.948462,0.5,pearson
1,net_g05_w55_low,ctx_w55_low,g05,0.5,55,low,376,3.177500,0.5,pearson
2,net_g05_w56_high,ctx_w56_high,g05,0.5,56,high,351,2.655842,0.5,pearson
3,net_g05_w56_low,ctx_w56_low,g05,0.5,56,low,383,3.048761,0.5,pearson
4,net_g05_w57_high,ctx_w57_high,g05,0.5,57,high,407,2.303442,0.5,pearson
5,net_g05_w57_low,ctx_w57_low,g05,0.5,57,low,336,3.265030,0.5,pearson


In [36]:
# ── Relationship table: HAS_CONTEXT ──────────────────────────────────────────
# GammaSpec has been removed from the schema. Estimation metadata (gamma,
# cor_method) is stored as properties on NetworkInstance.

rel_has_context = (
    network_instances[["network_id","context_id"]]
    .drop_duplicates()
    .sort_values(["network_id","context_id"])
    .reset_index(drop=True)
)

print("rel_has_context    :", rel_has_context.shape)


rel_has_context    : (6, 2)


In [37]:
# ── NodeInstance nodes and linking relationships ──────────────────────────────
# node_instance_id = network_id + "__" + node_id
# The double-underscore separator is safe because node names contain no "__".

node_instances = (
    node_metrics[node_metrics["spec"].isin(KG_SPECS)]
    [["network_id","context_id","node_id","node",
      "degree","strength","domain","role"]]
    .drop_duplicates()
    .assign(
        node_instance_id = lambda d: d["network_id"] + "__" + d["node_id"],
        # is_focal written as lowercase string so Neo4j toBoolean() works correctly.
        # Python bool → CSV → "True"/"False" but Neo4j toBoolean() requires "true"/"false".
        is_focal         = lambda d: d["role"].eq("focal_belief").map({True: "true", False: "false"}),
    )
    .sort_values(["network_id","domain","node_id"])
    .reset_index(drop=True)
)

if not node_instances["node_instance_id"].is_unique:
    raise ValueError("Duplicate node_instance_id values.")
if node_instances.duplicated(subset=["network_id","node_id"]).any():
    raise ValueError("Each variable appears more than once in at least one network instance.")

rel_network_has_node_instance = (
    node_instances[["network_id","node_instance_id"]]
    .drop_duplicates()
    .sort_values(["network_id","node_instance_id"])
    .reset_index(drop=True)
)

rel_node_instance_of_variable = (
    node_instances[["node_instance_id","node_id"]]
    .drop_duplicates()
    .sort_values(["node_instance_id","node_id"])
    .reset_index(drop=True)
)

print("node_instances                  :", node_instances.shape)
print("rel_network_has_node_instance   :", rel_network_has_node_instance.shape)
print("rel_node_instance_of_variable   :", rel_node_instance_of_variable.shape)
display(node_instances.head(10))

node_instances                  : (48, 10)
rel_network_has_node_instance   : (48, 2)
rel_node_instance_of_variable   : (48, 2)


,network_id,context_id,node_id,node,degree,strength,domain,role,node_instance_id,is_focal
0,net_g05_w55_high,ctx_w55_high,AFF_FEAR,AFF_FEAR,6,1.054480,affect,non_focal,net_g05_w55_high__AFF_FEAR,false
1,net_g05_w55_high,ctx_w55_high,AFF_WORRY,AFF_WORRY,5,0.779726,affect,non_focal,net_g05_w55_high__AFF_WORRY,false
2,net_g05_w55_high,ctx_w55_high,WORRY_HEALTH_SYSTEM,WORRY_HEALTH_SYSTEM,7,0.516938,affect,non_focal,net_g05_w55_high__WORRY_HEALTH_SYSTEM,false
3,net_g05_w55_high,ctx_w55_high,USE2_AVOID,USE2_AVOID,7,0.715792,behavior,non_focal,net_g05_w55_high__USE2_AVOID,false
4,net_g05_w55_high,ctx_w55_high,USE2_HANDWASH20,USE2_HANDWASH20,7,0.643894,behavior,non_focal,net_g05_w55_high__USE2_HANDWASH20,false
5,net_g05_w55_high,ctx_w55_high,USE2_MASK,USE2_MASK,6,0.626226,behavior,non_focal,net_g05_w55_high__USE2_MASK,false
6,net_g05_w55_high,ctx_w55_high,USE2_SPACE150,USE2_SPACE150,5,0.806979,behavior,non_focal,net_g05_w55_high__USE2_SPACE150,false
7,net_g05_w55_high,ctx_w55_high,SEVERITY,SEVERITY,7,0.752889,belief,focal_belief,net_g05_w55_high__SEVERITY,true
8,net_g05_w55_low,ctx_w55_low,AFF_FEAR,AFF_FEAR,4,1.023740,affect,non_focal,net_g05_w55_low__AFF_FEAR,false
9,net_g05_w55_low,ctx_w55_low,AFF_WORRY,AFF_WORRY,6,0.894734,affect,non_focal,net_g05_w55_low__AFF_WORRY,false


In [38]:
# ── EdgeInstance nodes and linking relationships ──────────────────────────────
# edge_instance_id = edge_id (already encodes network_id + edge_a + edge_b).
#
# FROM and TO relationships connect an EdgeInstance to its two endpoint
# NodeInstances.  Because GGM networks are undirected, FROM/TO here represent
# the two endpoints (canonically: from = alphabetic min, to = alphabetic max)
# and carry no directional interpretation.
#
# from_node_instance_id = network_id + "__" + from  (from = edge_a, lower node name)
# to_node_instance_id   = network_id + "__" + to    (to   = edge_b, upper node name)
#
# Bootstrap columns (sample_weight, boot_mean, etc.) are present for g05 and g075
# by design; all edges in edge_metrics_kg should have non-NA bootstrap values.
# A warning is emitted if any are missing.

edge_instances = (
    edge_metrics_kg[[
        "edge_id","network_id","context_id","spec_id",
        "from","to","edge_a","edge_b",
        "weight","abs_weight",
        "from_domain","to_domain","from_role","to_role",
        "sample_weight","boot_mean","boot_sd",
        "ci_low","ci_high","prop_nonzero","n_boot",
    ]]
    .drop_duplicates()
    .assign(
        edge_instance_id      = lambda d: d["edge_id"].astype(str),
        from_node_instance_id = lambda d: d["network_id"] + "__" + d["from"],
        to_node_instance_id   = lambda d: d["network_id"] + "__" + d["to"],
    )
    .sort_values(["network_id","abs_weight"], ascending=[True, False])
    .reset_index(drop=True)
)

if not edge_instances["edge_instance_id"].is_unique:
    raise ValueError("Duplicate edge_instance_id values.")

# Warn if bootstrap values are unexpectedly absent for KG specs
n_boot_na = edge_instances["boot_mean"].isna().sum()
if n_boot_na > 0:
    import warnings
    warnings.warn(
        f"{n_boot_na} edge instance(s) in KG specs have NA boot_mean. "
        "Bootstrap files may be missing for some contexts."
    )

# Referential integrity: every endpoint must exist in node_instances
valid_ni_ids = set(node_instances["node_instance_id"])
bad_from = ~edge_instances["from_node_instance_id"].isin(valid_ni_ids)
bad_to   = ~edge_instances["to_node_instance_id"].isin(valid_ni_ids)
if bad_from.any():
    raise ValueError(
        f"{bad_from.sum()} edge(s) have invalid from_node_instance_id.\n"
        + str(edge_instances.loc[bad_from, ["edge_instance_id","from_node_instance_id"]].head())
    )
if bad_to.any():
    raise ValueError(
        f"{bad_to.sum()} edge(s) have invalid to_node_instance_id.\n"
        + str(edge_instances.loc[bad_to, ["edge_instance_id","to_node_instance_id"]].head())
    )

rel_network_has_edge_instance = (
    edge_instances[["network_id","edge_instance_id"]]
    .drop_duplicates()
    .sort_values(["network_id","edge_instance_id"])
    .reset_index(drop=True)
)

rel_edge_instance_from = (
    edge_instances[["edge_instance_id","from_node_instance_id"]]
    .drop_duplicates()
    .rename(columns={"from_node_instance_id": "node_instance_id"})
    .sort_values(["edge_instance_id","node_instance_id"])
    .reset_index(drop=True)
)

rel_edge_instance_to = (
    edge_instances[["edge_instance_id","to_node_instance_id"]]
    .drop_duplicates()
    .rename(columns={"to_node_instance_id": "node_instance_id"})
    .sort_values(["edge_instance_id","node_instance_id"])
    .reset_index(drop=True)
)

print("edge_instances              :", edge_instances.shape)
print("rel_network_has_edge_instance:", rel_network_has_edge_instance.shape)
print("rel_edge_instance_from      :", rel_edge_instance_from.shape)
print("rel_edge_instance_to        :", rel_edge_instance_to.shape)

edge_instances              : (129, 24)
rel_network_has_edge_instance: (129, 2)
rel_edge_instance_from      : (129, 2)
rel_edge_instance_to        : (129, 2)


In [39]:
# ── Schema validation ─────────────────────────────────────────────────────────
N_VARIABLES_EXPECTED = 8   # 1 belief + 3 affect + 4 behavior

if len(variables) != N_VARIABLES_EXPECTED:
    raise ValueError(
        f"Expected {N_VARIABLES_EXPECTED} variables, found {len(variables)}."
    )

# Every network instance must contain all variables exactly once
node_counts = node_instances.groupby("network_id")["node_id"].nunique()
wrong_counts = node_counts[node_counts != N_VARIABLES_EXPECTED]
if not wrong_counts.empty:
    raise ValueError(
        "The following network instances do not contain exactly "
        f"{N_VARIABLES_EXPECTED} nodes:\n" + str(wrong_counts)
    )

# Edge count range (diagnostic only — not a hard constraint)
edge_counts = edge_instances.groupby("network_id")["edge_instance_id"].nunique()
max_edges   = N_VARIABLES_EXPECTED * (N_VARIABLES_EXPECTED - 1) // 2   # 28

if (edge_counts > max_edges).any():
    raise ValueError(
        f"Some network instances have more than {max_edges} edges (impossible for "
        f"{N_VARIABLES_EXPECTED} nodes). Check for duplicate edges."
    )

print(f"Variables         : {len(variables)}  (expected {N_VARIABLES_EXPECTED})")
print(f"Contexts          : {len(contexts)}")
print(f"Network instances : {len(network_instances)}  (one per context)")
print(f"Node instances    : {len(node_instances)}")
print(f"Edge instances    : {len(edge_instances)}")
print(f"Edges per network : min={edge_counts.min()}, max={edge_counts.max()} (max possible={max_edges})")
print("\nAll schema checks passed ✓")


Variables         : 8  (expected 8)
Contexts          : 6
Network instances : 6  (one per context)
Node instances    : 48
Edge instances    : 129
Edges per network : min=17, max=25 (max possible=28)

All schema checks passed ✓


In [40]:
# ── Export all KG tables as CSV ───────────────────────────────────────────────
EXPORT_MAP = {
    # Node tables
    "variables.csv"        : variables,
    "contexts.csv"         : contexts,
    "network_instances.csv": network_instances,
    "node_instances.csv"   : node_instances,
    "edge_instances.csv"   : edge_instances,
    # Relationship tables
    "rel_has_context.csv"                 : rel_has_context,
    "rel_network_has_node_instance.csv"   : rel_network_has_node_instance,
    "rel_node_instance_of_variable.csv"   : rel_node_instance_of_variable,
    "rel_network_has_edge_instance.csv"   : rel_network_has_edge_instance,
    "rel_edge_instance_from.csv"          : rel_edge_instance_from,
    "rel_edge_instance_to.csv"            : rel_edge_instance_to,
}

for fname, df in EXPORT_MAP.items():
    df.to_csv(KG_DIR / fname, index=False)
    print(f"Saved: {fname:<45} {df.shape}")

print(f"\nAll KG tables exported to: {KG_DIR.resolve()}")
print("Note: gamma_specs.csv and rel_has_gamma_spec.csv are no longer exported.")


Saved: variables.csv                                 (8, 4)
Saved: contexts.csv                                  (6, 7)
Saved: network_instances.csv                         (6, 10)
Saved: node_instances.csv                            (48, 10)
Saved: edge_instances.csv                            (129, 24)
Saved: rel_has_context.csv                           (6, 2)
Saved: rel_network_has_node_instance.csv             (48, 2)
Saved: rel_node_instance_of_variable.csv             (48, 2)
Saved: rel_network_has_edge_instance.csv             (129, 2)
Saved: rel_edge_instance_from.csv                    (129, 2)
Saved: rel_edge_instance_to.csv                      (129, 2)

All KG tables exported to: C:\Users\Ciinz\Desktop\ThesisFinal\Results\networks\kg_exports
Note: gamma_specs.csv and rel_has_gamma_spec.csv are no longer exported.


## Neo4j Cypher import script

### Key implementation notes

1. **`from` is a reserved Cypher keyword.** All references to the `from` property on
   `EdgeInstance` nodes must be backtick-escaped as `` `from` ``.

2. **Boolean handling.** Python's `True`/`False` serializes to `"True"`/`"False"` in CSV.
   Neo4j's `toBoolean()` requires lowercase `"true"`/`"false"`.  
   The `is_focal` column is written as lowercase strings (`"true"`/`"false"`) in Cell 7
   above, so `toBoolean(row.is_focal)` will work correctly.

3. **Nullable float/integer columns.** Bootstrap columns (`boot_mean`, `boot_sd`,
   `ci_low`, `ci_high`, `prop_nonzero`, `n_boot`, `sample_weight`) may be empty strings
   in CSV when the value is missing.  Neo4j's `toFloat("")` and `toInteger("")` return
   `null` in version 4.4+ but may throw in earlier versions.  The Cypher uses
   `CASE WHEN row.X <> '' THEN toFloat(row.X) END` for all nullable columns.

4. **Idempotency.** All imports use `MERGE`, so the script can be re-run safely.

5. **Constraints must be created before `MERGE` statements.** The Cypher script
   creates all constraints first; each `MERGE` then uses the constrained property
   as the lookup key.

In [41]:
# ── Generate Neo4j Cypher import script ──────────────────────────────────────
# Notes:
# - 'from' is backtick-escaped throughout (reserved Cypher keyword).
# Estimation metadata stored as properties on NetworkInstance.
# - Nullable bootstrap columns use CASE WHEN to handle empty-string CSV values
#   safely across Neo4j versions 4.4 and 5.x.
# - is_focal is stored as lowercase string in CSV (see Cell 7), so toBoolean() works.

CYPHER = """
// ==========================================================================
// BKG import script — generated by 04_kg_encoding
// Place CSV files in Neo4j's import folder before running.
// Compatible with Neo4j 4.4+ and 5.x.
// Schema: 6 NetworkInstances (one per context), no GammaSpec nodes.
// ==========================================================================

// --------------------------------------------------------------------------
// Constraints (must be created before MERGE statements)
// --------------------------------------------------------------------------
CREATE CONSTRAINT variable_node_id      IF NOT EXISTS FOR (n:Variable)       REQUIRE n.node_id          IS UNIQUE;
CREATE CONSTRAINT context_context_id    IF NOT EXISTS FOR (n:Context)         REQUIRE n.context_id       IS UNIQUE;
CREATE CONSTRAINT network_network_id    IF NOT EXISTS FOR (n:NetworkInstance) REQUIRE n.network_id       IS UNIQUE;
CREATE CONSTRAINT node_instance_id      IF NOT EXISTS FOR (n:NodeInstance)    REQUIRE n.node_instance_id IS UNIQUE;
CREATE CONSTRAINT edge_instance_id      IF NOT EXISTS FOR (n:EdgeInstance)    REQUIRE n.edge_instance_id IS UNIQUE;

// --------------------------------------------------------------------------
// Variable nodes
// --------------------------------------------------------------------------
LOAD CSV WITH HEADERS FROM 'file:///variables.csv' AS row
MERGE (v:Variable {node_id: row.node_id})
SET
  v.name   = row.name,
  v.domain = row.domain,
  v.role   = row.role;

// --------------------------------------------------------------------------
// Context nodes
// --------------------------------------------------------------------------
LOAD CSV WITH HEADERS FROM 'file:///contexts.csv' AS row
MERGE (c:Context {context_id: row.context_id})
SET
  c.wave                    = toInteger(row.wave),
  c.trust_stratum           = row.trust_stratum,
  c.n_cc                    = toInteger(row.n_cc),
  c.trust_variable          = row.trust_variable,
  c.trust_grouping_rule     = row.trust_grouping_rule,
  c.middle_tercile_excluded = toBoolean(row.middle_tercile_excluded);

// --------------------------------------------------------------------------
// NetworkInstance nodes
// Estimation metadata (gamma, cor_method) stored as properties.
// --------------------------------------------------------------------------
LOAD CSV WITH HEADERS FROM 'file:///network_instances.csv' AS row
MERGE (n:NetworkInstance {network_id: row.network_id})
SET
  n.context_id      = row.context_id,
  n.spec            = row.spec,
  n.gamma_ebic      = toFloat(row.gamma_ebic),
  n.gamma           = toFloat(row.gamma),
  n.cor_method      = row.cor_method,
  n.wave            = toInteger(row.wave),
  n.trust_stratum   = row.trust_stratum,
  n.n_cc            = toInteger(row.n_cc),
  n.global_strength = toFloat(row.global_strength);

// --------------------------------------------------------------------------
// NodeInstance nodes
// --------------------------------------------------------------------------
LOAD CSV WITH HEADERS FROM 'file:///node_instances.csv' AS row
MERGE (ni:NodeInstance {node_instance_id: row.node_instance_id})
SET
  ni.network_id = row.network_id,
  ni.context_id = row.context_id,
  ni.node_id    = row.node_id,
  ni.node       = row.node,
  ni.degree     = toInteger(row.degree),
  ni.strength   = toFloat(row.strength),
  ni.domain     = row.domain,
  ni.role       = row.role,
  ni.is_focal   = toBoolean(row.is_focal);

// --------------------------------------------------------------------------
// EdgeInstance nodes
// 'from' is backtick-escaped — it is a reserved Cypher keyword.
// Nullable bootstrap columns use CASE WHEN to handle empty-string CSV values.
// --------------------------------------------------------------------------
LOAD CSV WITH HEADERS FROM 'file:///edge_instances.csv' AS row
MERGE (ei:EdgeInstance {edge_instance_id: row.edge_instance_id})
SET
  ei.network_id   = row.network_id,
  ei.context_id   = row.context_id,
  ei.`from`       = row.`from`,
  ei.to           = row.to,
  ei.edge_a       = row.edge_a,
  ei.edge_b       = row.edge_b,
  ei.weight       = toFloat(row.weight),
  ei.abs_weight   = toFloat(row.abs_weight),
  ei.from_domain  = row.from_domain,
  ei.to_domain    = row.to_domain,
  ei.from_role    = row.from_role,
  ei.to_role      = row.to_role,
  ei.sample_weight = CASE WHEN row.sample_weight <> '' THEN toFloat(row.sample_weight)  END,
  ei.boot_mean     = CASE WHEN row.boot_mean     <> '' THEN toFloat(row.boot_mean)      END,
  ei.boot_sd       = CASE WHEN row.boot_sd       <> '' THEN toFloat(row.boot_sd)        END,
  ei.ci_low        = CASE WHEN row.ci_low        <> '' THEN toFloat(row.ci_low)         END,
  ei.ci_high       = CASE WHEN row.ci_high       <> '' THEN toFloat(row.ci_high)        END,
  ei.prop_nonzero  = CASE WHEN row.prop_nonzero  <> '' THEN toFloat(row.prop_nonzero)   END,
  ei.n_boot        = CASE WHEN row.n_boot        <> '' THEN toInteger(row.n_boot)       END;

// --------------------------------------------------------------------------
// Relationships
// --------------------------------------------------------------------------

// NetworkInstance -[:HAS_CONTEXT]-> Context
LOAD CSV WITH HEADERS FROM 'file:///rel_has_context.csv' AS row
MATCH (n:NetworkInstance {network_id: row.network_id})
MATCH (c:Context         {context_id: row.context_id})
MERGE (n)-[:HAS_CONTEXT]->(c);

// NetworkInstance -[:HAS_NODE_INSTANCE]-> NodeInstance
LOAD CSV WITH HEADERS FROM 'file:///rel_network_has_node_instance.csv' AS row
MATCH (n:NetworkInstance {network_id:       row.network_id})
MATCH (ni:NodeInstance   {node_instance_id: row.node_instance_id})
MERGE (n)-[:HAS_NODE_INSTANCE]->(ni);

// NodeInstance -[:INSTANCE_OF]-> Variable
LOAD CSV WITH HEADERS FROM 'file:///rel_node_instance_of_variable.csv' AS row
MATCH (ni:NodeInstance {node_instance_id: row.node_instance_id})
MATCH (v:Variable      {node_id:          row.node_id})
MERGE (ni)-[:INSTANCE_OF]->(v);

// NetworkInstance -[:HAS_EDGE_INSTANCE]-> EdgeInstance
LOAD CSV WITH HEADERS FROM 'file:///rel_network_has_edge_instance.csv' AS row
MATCH (n:NetworkInstance {network_id:        row.network_id})
MATCH (ei:EdgeInstance   {edge_instance_id:  row.edge_instance_id})
MERGE (n)-[:HAS_EDGE_INSTANCE]->(ei);

// EdgeInstance -[:FROM]-> NodeInstance  (endpoint, not directed)
LOAD CSV WITH HEADERS FROM 'file:///rel_edge_instance_from.csv' AS row
MATCH (ei:EdgeInstance {edge_instance_id: row.edge_instance_id})
MATCH (ni:NodeInstance {node_instance_id: row.node_instance_id})
MERGE (ei)-[:FROM]->(ni);

// EdgeInstance -[:TO]-> NodeInstance  (endpoint, not directed)
LOAD CSV WITH HEADERS FROM 'file:///rel_edge_instance_to.csv' AS row
MATCH (ei:EdgeInstance {edge_instance_id: row.edge_instance_id})
MATCH (ni:NodeInstance {node_instance_id: row.node_instance_id})
MERGE (ei)-[:TO]->(ni);
""".strip()

cypher_path = KG_DIR / "import_kg.cypher"
cypher_path.write_text(CYPHER, encoding="utf-8")
print("Saved:", cypher_path.resolve())


Saved: C:\Users\Ciinz\Desktop\ThesisFinal\Results\networks\kg_exports\import_kg.cypher


In [42]:
# ── Preview all KG tables ─────────────────────────────────────────────────────
_PREVIEWS = {
    "Variables"                      : variables,
    "Contexts"                       : contexts,
    "Network instances"              : network_instances,
    "Node instances (first 10)"      : node_instances.head(10),
    "Edge instances (first 10)"      : edge_instances.head(10),
    "rel_has_context"                : rel_has_context,
    "rel_network_has_node_instance"  : rel_network_has_node_instance.head(10),
    "rel_node_instance_of_variable"  : rel_node_instance_of_variable.head(10),
    "rel_network_has_edge_instance"  : rel_network_has_edge_instance.head(10),
    "rel_edge_instance_from"         : rel_edge_instance_from.head(10),
    "rel_edge_instance_to"           : rel_edge_instance_to.head(10),
}

for label, df in _PREVIEWS.items():
    print(f"\n{'─'*60}")
    print(label)
    display(df)



────────────────────────────────────────────────────────────
Variables


,node_id,name,domain,role
0,AFF_FEAR,AFF_FEAR,affect,non_focal
1,AFF_WORRY,AFF_WORRY,affect,non_focal
2,WORRY_HEALTH_SYSTEM,WORRY_HEALTH_SYSTEM,affect,non_focal
3,USE2_AVOID,USE2_AVOID,behavior,non_focal
4,USE2_HANDWASH20,USE2_HANDWASH20,behavior,non_focal
5,USE2_MASK,USE2_MASK,behavior,non_focal
6,USE2_SPACE150,USE2_SPACE150,behavior,non_focal
7,SEVERITY,SEVERITY,belief,focal_belief



────────────────────────────────────────────────────────────
Contexts


,context_id,wave,trust_stratum,n_cc,trust_variable,trust_grouping_rule,middle_tercile_excluded
0,ctx_w55_high,55,high,309,TRUST_RKI,within-wave terciles,True
1,ctx_w55_low,55,low,376,TRUST_RKI,within-wave terciles,True
2,ctx_w56_high,56,high,351,TRUST_RKI,within-wave terciles,True
3,ctx_w56_low,56,low,383,TRUST_RKI,within-wave terciles,True
4,ctx_w57_high,57,high,407,TRUST_RKI,within-wave terciles,True
5,ctx_w57_low,57,low,336,TRUST_RKI,within-wave terciles,True



────────────────────────────────────────────────────────────
Network instances


...

,network_id,edge_instance_id
0,net_g05_w55_high,net_g05_w55_high__AFF_FEAR__AFF_WORRY
1,net_g05_w55_high,net_g05_w55_high__AFF_FEAR__SEVERITY
2,net_g05_w55_high,net_g05_w55_high__AFF_FEAR__USE2_AVOID
3,net_g05_w55_high,net_g05_w55_high__AFF_FEAR__USE2_HANDWASH20
4,net_g05_w55_high,net_g05_w55_high__AFF_FEAR__USE2_MASK
5,net_g05_w55_high,net_g05_w55_high__AFF_FEAR__WORRY_HEALTH_SYSTEM
6,net_g05_w55_high,net_g05_w55_high__AFF_WORRY__SEVERITY
7,net_g05_w55_high,net_g05_w55_high__AFF_WORRY__USE2_AVOID
8,net_g05_w55_high,net_g05_w55_high__AFF_WORRY__USE2_HANDWASH20
9,net_g05_w55_high,net_g05_w55_high__AFF_WORRY__WORRY_HEALTH_SYSTEM



────────────────────────────────────────────────────────────
rel_edge_instance_from


,edge_instance_id,node_instance_id
0,net_g05_w55_high__AFF_FEAR__AFF_WORRY,net_g05_w55_high__AFF_FEAR
1,net_g05_w55_high__AFF_FEAR__SEVERITY,net_g05_w55_high__AFF_FEAR
2,net_g05_w55_high__AFF_FEAR__USE2_AVOID,net_g05_w55_high__AFF_FEAR
3,net_g05_w55_high__AFF_FEAR__USE2_HANDWASH20,net_g05_w55_high__AFF_FEAR
4,net_g05_w55_high__AFF_FEAR__USE2_MASK,net_g05_w55_high__AFF_FEAR
5,net_g05_w55_high__AFF_FEAR__WORRY_HEALTH_SYSTEM,net_g05_w55_high__AFF_FEAR
6,net_g05_w55_high__AFF_WORRY__SEVERITY,net_g05_w55_high__AFF_WORRY
7,net_g05_w55_high__AFF_WORRY__USE2_AVOID,net_g05_w55_high__AFF_WORRY
8,net_g05_w55_high__AFF_WORRY__USE2_HANDWASH20,net_g05_w55_high__AFF_WORRY
9,net_g05_w55_high__AFF_WORRY__WORRY_HEALTH_SYSTEM,net_g05_w55_high__AFF_WORRY



────────────────────────────────────────────────────────────
rel_edge_instance_to


,edge_instance_id,node_instance_id
0,net_g05_w55_high__AFF_FEAR__AFF_WORRY,net_g05_w55_high__AFF_WORRY
1,net_g05_w55_high__AFF_FEAR__SEVERITY,net_g05_w55_high__SEVERITY
2,net_g05_w55_high__AFF_FEAR__USE2_AVOID,net_g05_w55_high__USE2_AVOID
3,net_g05_w55_high__AFF_FEAR__USE2_HANDWASH20,net_g05_w55_high__USE2_HANDWASH20
4,net_g05_w55_high__AFF_FEAR__USE2_MASK,net_g05_w55_high__USE2_MASK
5,net_g05_w55_high__AFF_FEAR__WORRY_HEALTH_SYSTEM,net_g05_w55_high__WORRY_HEALTH_SYSTEM
6,net_g05_w55_high__AFF_WORRY__SEVERITY,net_g05_w55_high__SEVERITY
7,net_g05_w55_high__AFF_WORRY__USE2_AVOID,net_g05_w55_high__USE2_AVOID
8,net_g05_w55_high__AFF_WORRY__USE2_HANDWASH20,net_g05_w55_high__USE2_HANDWASH20
9,net_g05_w55_high__AFF_WORRY__WORRY_HEALTH_SYSTEM,net_g05_w55_high__WORRY_HEALTH_SYSTEM


In [43]:
# ── Available network instances ───────────────────────────────────────────────
available_instances = (
    network_instances[["network_id","wave","trust_stratum","gamma_ebic","spec","n_cc","global_strength"]]
    .sort_values(["wave","trust_stratum","gamma_ebic"])
    .reset_index(drop=True)
)
print("Network instances available for querying:")
display(available_instances)

Network instances available for querying:


,network_id,wave,trust_stratum,gamma_ebic,spec,n_cc,global_strength
0,net_g05_w55_high,55,high,0.5,g05,309,2.948462
1,net_g05_w55_low,55,low,0.5,g05,376,3.177500
2,net_g05_w56_high,56,high,0.5,g05,351,2.655842
3,net_g05_w56_low,56,low,0.5,g05,383,3.048761
4,net_g05_w57_high,57,high,0.5,g05,407,2.303442
5,net_g05_w57_low,57,low,0.5,g05,336,3.265030


In [44]:
# ── Example: preview one network instance as it will appear in the KG ─────────
EXAMPLE_WAVE  = 55
EXAMPLE_TRUST = "high"
EXAMPLE_GAMMA = 0.5

mask = (
    (network_instances["wave"]           == EXAMPLE_WAVE)  &
    (network_instances["trust_stratum"]  == EXAMPLE_TRUST) &
    (network_instances["gamma_ebic"]     == EXAMPLE_GAMMA)
)
matches = network_instances.loc[mask, "network_id"]
if matches.empty:
    raise ValueError(
        f"No network instance found for wave={EXAMPLE_WAVE}, "
        f"trust={EXAMPLE_TRUST}, gamma={EXAMPLE_GAMMA}."
    )
example_network = matches.iloc[0]

example_nodes = (
    node_instances.loc[node_instances["network_id"] == example_network]
    .merge(variables[["node_id","name"]], on="node_id", how="left")
    .sort_values("strength", ascending=False)
    .reset_index(drop=True)
)

example_edges = (
    edge_instances.loc[edge_instances["network_id"] == example_network]
    .sort_values("abs_weight", ascending=False)
    .reset_index(drop=True)
)

print(f"Selected network : {example_network}")
print(f"\nNodeInstances ({len(example_nodes)} nodes):")
display(example_nodes)
print(f"\nEdgeInstances ({len(example_edges)} edges, top 10 by |weight|):")
display(example_edges.head(10))

Selected network : net_g05_w55_high

NodeInstances (8 nodes):


,network_id,context_id,node_id,node,degree,strength,domain,role,node_instance_id,is_focal,name
0,net_g05_w55_high,ctx_w55_high,AFF_FEAR,AFF_FEAR,6,1.054480,affect,non_focal,net_g05_w55_high__AFF_FEAR,false,AFF_FEAR
1,net_g05_w55_high,ctx_w55_high,USE2_SPACE150,USE2_SPACE150,5,0.806979,behavior,non_focal,net_g05_w55_high__USE2_SPACE150,false,USE2_SPACE150
2,net_g05_w55_high,ctx_w55_high,AFF_WORRY,AFF_WORRY,5,0.779726,affect,non_focal,net_g05_w55_high__AFF_WORRY,false,AFF_WORRY
3,net_g05_w55_high,ctx_w55_high,SEVERITY,SEVERITY,7,0.752889,belief,focal_belief,net_g05_w55_high__SEVERITY,true,SEVERITY
4,net_g05_w55_high,ctx_w55_high,USE2_AVOID,USE2_AVOID,7,0.715792,behavior,non_focal,net_g05_w55_high__USE2_AVOID,false,USE2_AVOID
5,net_g05_w55_high,ctx_w55_high,USE2_HANDWASH20,USE2_HANDWASH20,7,0.643894,behavior,non_focal,net_g05_w55_high__USE2_HANDWASH20,false,USE2_HANDWASH20
6,net_g05_w55_high,ctx_w55_high,USE2_MASK,USE2_MASK,6,0.626226,behavior,non_focal,net_g05_w55_high__USE2_MASK,false,USE2_MASK
7,net_g05_w55_high,ctx_w55_high,WORRY_HEALTH_SYSTEM,WORRY_HEALTH_SYSTEM,7,0.516938,affect,non_focal,net_g05_w55_high__WORRY_HEALTH_SYSTEM,false,WORRY_HEALTH_SYSTEM



EdgeInstances (25 edges, top 10 by |weight|):


,edge_id,network_id,context_id,spec_id,from,to,edge_a,edge_b,weight,abs_weight,from_domain,to_domain,from_role,to_role,sample_weight,boot_mean,boot_sd,ci_low,ci_high,prop_nonzero,n_boot,edge_instance_id,from_node_instance_id,to_node_instance_id
0,net_g05_w55_high__AFF_FEAR__AFF_WORRY,net_g05_w55_high,ctx_w55_high,gamma_05,AFF_FEAR,AFF_WORRY,AFF_FEAR,AFF_WORRY,0.577395,0.577395,affect,affect,non_focal,non_focal,0.577395,0.560125,0.050116,0.456583,0.646848,1.000,1000.0,net_g05_w55_high__AFF_FEAR__AFF_WORRY,net_g05_w55_high__AFF_FEAR,net_g05_w55_high__AFF_WORRY
1,net_g05_w55_high__USE2_AVOID__USE2_SPACE150,net_g05_w55_high,ctx_w55_high,gamma_05,USE2_AVOID,USE2_SPACE150,USE2_AVOID,USE2_SPACE150,0.237941,0.237941,behavior,behavior,non_focal,non_focal,0.237941,0.232918,0.052970,0.123881,0.336130,1.000,1000.0,net_g05_w55_high__USE2_AVOID__USE2_SPACE150,net_g05_w55_high__USE2_AVOID,net_g05_w55_high__USE2_SPACE150
2,net_g05_w55_high__USE2_MASK__USE2_SPACE150,net_g05_w55_high,ctx_w55_high,gamma_05,USE2_MASK,USE2_SPACE150,USE2_MASK,USE2_SPACE150,0.213591,0.213591,behavior,behavior,non_focal,non_focal,0.213591,0.200779,0.065500,0.066358,0.319884,0.997,1000.0,net_g05_w55_high__USE2_MASK__USE2_SPACE150,net_g05_w55_high__USE2_MASK,net_g05_w55_high__USE2_SPACE150
3,net_g05_w55_high__USE2_HANDWASH20__USE2_SPACE150,net_g05_w55_high,ctx_w55_high,gamma_05,USE2_HANDWASH20,USE2_SPACE150,USE2_HANDWASH20,USE2_SPACE150,0.203364,0.203364,behavior,behavior,non_focal,non_focal,0.203364,0.197165,0.060766,0.075266,0.316154,1.000,1000.0,net_g05_w55_high__USE2_HANDWASH20__USE2_SPACE150,net_g05_w55_high__USE2_HANDWASH20,net_g05_w55_high__USE2_SPACE150
4,net_g05_w55_high__AFF_FEAR__SEVERITY,net_g05_w55_high,ctx_w55_high,gamma_05,AFF_FEAR,SEVERITY,AFF_FEAR,SEVERITY,0.196328,0.196328,affect,belief,non_focal,focal_belief,0.196328,0.199123,0.049647,0.092539,0.291625,1.000,1000.0,net_g05_w55_high__AFF_FEAR__SEVERITY,net_g05_w55_high__AFF_FEAR,net_g05_w55_high__SEVERITY
5,net_g05_w55_high__USE2_HANDWASH20__USE2_MASK,net_g05_w55_high,ctx_w55_high,gamma_05,USE2_HANDWASH20,USE2_MASK,USE2_HANDWASH20,USE2_MASK,0.183262,0.183262,behavior,behavior,non_focal,non_focal,0.183262,0.170599,0.057138,0.054651,0.283808,0.995,1000.0,net_g05_w55_high__USE2_HANDWASH20__USE2_MASK,net_g05_w55_high__USE2_HANDWASH20,net_g05_w55_high__USE2_MASK
6,net_g05_w55_high__AFF_FEAR__WORRY_HEALTH_SYSTEM,net_g05_w55_high,ctx_w55_high,gamma_05,AFF_FEAR,WORRY_HEALTH_SYSTEM,AFF_FEAR,WORRY_HEALTH_SYSTEM,0.177962,0.177962,affect,affect,non_focal,non_focal,0.177962,0.173719,0.058309,0.053726,0.280572,0.999,1000.0,net_g05_w55_high__AFF_FEAR__WORRY_HEALTH_SYSTEM,net_g05_w55_high__AFF_FEAR,net_g05_w55_high__WORRY_HEALTH_SYSTEM
7,net_g05_w55_high__USE2_AVOID__USE2_HANDWASH20,net_g05_w55_high,ctx_w55_high,gamma_05,USE2_AVOID,USE2_HANDWASH20,USE2_AVOID,USE2_HANDWASH20,0.144586,0.144586,behavior,behavior,non_focal,non_focal,0.144586,0.139636,0.052909,0.031492,0.242295,0.995,1000.0,net_g05_w55_high__USE2_AVOID__USE2_HANDWASH20,net_g05_w55_high__USE2_AVOID,net_g05_w55_high__USE2_HANDWASH20
8,net_g05_w55_high__SEVERITY__USE2_AVOID,net_g05_w55_high,ctx_w55_high,gamma_05,SEVERITY,USE2_AVOID,SEVERITY,USE2_AVOID,0.140764,0.140764,belief,behavior,focal_belief,non_focal,0.140764,0.131176,0.052128,0.024782,0.234604,0.989,1000.0,net_g05_w55_high__SEVERITY__USE2_AVOID,net_g05_w55_high__SEVERITY,net_g05_w55_high__USE2_AVOID
9,net_g05_w55_high__SEVERITY__USE2_SPACE150,net_g05_w55_high,ctx_w55_high,gamma_05,SEVERITY,USE2_SPACE150,SEVERITY,USE2_SPACE150,0.127414,0.127414,belief,behavior,focal_belief,non_focal,0.127414,0.122067,0.056560,0.000000,0.235357,0.972,1000.0,net_g05_w55_high__SEVERITY__USE2_SPACE150,net_g05_w55_high__SEVERITY,net_g05_w55_high__USE2_SPACE150


In [45]:
# ── Final file verification ───────────────────────────────────────────────────
csv_files = sorted(KG_DIR.glob("*.csv"))
print(f"CSV files in {KG_DIR.name}/:")
for f in csv_files:
    df = pd.read_csv(f)
    print(f"  {f.name:<50} shape={df.shape}  cols={df.columns.tolist()[:5]}{'...' if len(df.columns)>5 else ''}")

cypher_file = KG_DIR / "import_kg.cypher"
print(f"\nCypher script : {cypher_file.name}  ({cypher_file.stat().st_size:,} bytes)")

CSV files in kg_exports/:
  contexts.csv                                       shape=(6, 7)  cols=['context_id', 'wave', 'trust_stratum', 'n_cc', 'trust_variable']...
  edge_instances.csv                                 shape=(129, 24)  cols=['edge_id', 'network_id', 'context_id', 'spec_id', 'from']...
  network_instances.csv                              shape=(6, 10)  cols=['network_id', 'context_id', 'spec', 'gamma_ebic', 'wave']...
  node_instances.csv                                 shape=(48, 10)  cols=['network_id', 'context_id', 'node_id', 'node', 'degree']...
  rel_edge_instance_from.csv                         shape=(129, 2)  cols=['edge_instance_id', 'node_instance_id']
  rel_edge_instance_to.csv                           shape=(129, 2)  cols=['edge_instance_id', 'node_instance_id']
  rel_has_context.csv                                shape=(6, 2)  cols=['network_id', 'context_id']
  rel_network_has_edge_instance.csv                  shape=(129, 2)  cols=['network_id', 'edge_